# Food Delivery Time Prediction with XGBoost

Predicts food delivery time using gradient boosted trees (XGBoost). The pipeline covers EDA, feature engineering, hyperparameter search via random grid search, and final submission generation.

## 0. Setup

In [ ]:
!pip install -q xgboost scikit-learn pandas matplotlib

In [ ]:
import os
import random
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor

In [ ]:
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_SUB_PATH = "sample_submission.csv"
OUTPUT_SUB_PATH = "submission.csv"

CAT_COL = "weather"
NUM_COL = "rider_rating"
TARGET_COL = "delivery_time"

COORDINATE_COLS = [
    "restaurant_latitude", "restaurant_longitude",
    "delivery_latitude",   "delivery_longitude",
]

ALL_FEATURES = [
    CAT_COL, NUM_COL, TARGET_COL,
    "order_time", "pickup_time", "traffic_density",
    "is_festival", "vehicle_type", "vehicle_condition",
    "num_orders", "city", "order_date", "order_type", "rider_age",
] + COORDINATE_COLS

## 1. Exploratory Data Analysis

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

### 1.1 Numeric columns — missing values & distribution

In [ ]:
total_rows = len(train_df)

rating = train_df[NUM_COL]
print(f"Missing count ({NUM_COL}): {rating.isna().sum()}")
print(f"Missing ratio ({NUM_COL}): {rating.isna().mean():.4f}")
print()
print(rating.describe())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

rating_clean = rating.dropna()
mean_val = rating_clean.mean()
median_val = rating_clean.median()

ax1.hist(rating_clean, bins=30)
ax1.axvline(mean_val, color="r", linestyle="--", label=f"Mean: {mean_val:.2f}")
ax1.axvline(median_val, color="g", linestyle="--", label=f"Median: {median_val:.2f}")
ax1.set_title("Distribution of Rider Rating")
ax1.set_xlabel("Rating")
ax1.set_ylabel("Frequency")
ax1.legend()

ax2.boxplot(rating_clean, vert=True)
ax2.set_title("Boxplot of Rider Rating")

plt.tight_layout()
plt.show()

### 1.2 Coordinate columns — invalid value check

In [ ]:
for col in COORDINATE_COLS:
    bad = ((train_df[col].isna()) | (train_df[col] < 1)).sum()
    print(f"{col:35s}  bad rows: {bad}  ({bad/total_rows:.4f})")

### 1.3 Categorical columns — unique categories & missing values

In [ ]:
cat_cols = ["weather", "traffic_density", "city", "order_type", "is_festival", "vehicle_type"]

for col in cat_cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(train_df[col].value_counts(dropna=False))

## 2. Preprocessing & Feature Engineering

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    r = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return r * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


WEATHER_SEVERITY = {
    "clear_sky": 1,
    "foggy": 2,
    "overcast": 2,
    "high_wind": 3,
    "sandstorm": 4,
    "storm": 4,
}

TRAFFIC_WEIGHT = {"jam": 4, "high": 3, "medium": 2, "low": 1}
FESTIVAL_MAP = {"yes": 1, "y": 1, "no": 0, "x": 0, "nan": 0}


def preprocess(df, stats, is_train=True):
    df = df.copy()

    # ── datetime features ─────────────────────────────────────────────────
    df["order_time"] = df["order_time"].str.strip()
    df["pickup_time"] = df["pickup_time"].str.strip()

    df["order_time_dt"] = pd.to_datetime(df["order_time"],  errors="coerce")
    df["pickup_time_dt"] = pd.to_datetime(df["pickup_time"], errors="coerce")

    df["pickup_time_dt"] = df["pickup_time_dt"].fillna(
        df["order_time_dt"] + pd.Timedelta(minutes=10)
    )
    df["prep_time_min"] = (
        (df["pickup_time_dt"] - df["order_time_dt"]).dt.total_seconds() / 60
    )

    df["order_date_dt"] = pd.to_datetime(
        df["order_date"].astype(str).str.strip(), errors="coerce"
    )

    df["order_time_dt"] = df["order_time_dt"].fillna(stats["time_mode"])
    df["order_date_dt"] = df["order_date_dt"].fillna(stats["date_mode"])

    df["order_hour"] = df["order_time_dt"].dt.hour
    df["order_min"] = df["order_time_dt"].dt.minute
    df["day_of_week"] = df["order_date_dt"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    # ── categorical imputation ────────────────────────────────────────────
    df["is_festival"] = (
        df["is_festival"].astype(str).str.strip().str.lower()
    )
    df["is_festival_num"] = df["is_festival"].map(FESTIVAL_MAP).fillna(0)

    df["traffic_density"] = df.groupby("order_hour")["traffic_density"].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "medium")
    )
    df["traffic_intensity"] = df["traffic_density"].map(TRAFFIC_WEIGHT).fillna(2)

    df[CAT_COL] = df[CAT_COL].fillna(stats["mode_cat"])
    df["city"] = df["city"].fillna(stats["mode_city"])
    df["num_orders"] = df["num_orders"].fillna(1)
    df["vehicle_condition"] = df["vehicle_condition"].fillna(
        df["vehicle_condition"].median()
    )

    # ── numeric imputation + missing indicator ────────────────────────────
    df[f"{NUM_COL}_missing"] = df[NUM_COL].isna().astype(int)
    df[f"{NUM_COL}_imputed"] = df[NUM_COL].fillna(stats["median_num"])

    df["rider_age_missing"] = df["rider_age"].isna().astype(int)
    df["rider_age_imputed"] = df["rider_age"].fillna(stats["median_rider_age"])

    # ── distance ──────────────────────────────────────────────────────────
    df["distance_km"] = haversine_distance(
        df["restaurant_latitude"], df["restaurant_longitude"],
        df["delivery_latitude"], df["delivery_longitude"],
    )

    # ── time-of-day flags ─────────────────────────────────────────────────
    h = df["order_hour"]
    df["is_rush_hour"] = (((h >= 7) & (h <= 9)) | ((h >= 17) & (h <= 20))).astype(int)
    df["is_late_night"] = ((h >= 22) | (h <= 5)).astype(int)
    df["is_lunch_time"] = ((h >= 11) & (h <= 14)).astype(int)
    df["is_dinner_time"] = ((h >= 18) & (h <= 21)).astype(int)

    # ── interaction & derived features ────────────────────────────────────
    df["distance_bin"] = pd.cut(
        df["distance_km"],
        bins=[0, 2, 5, 10, 20, 100],
        labels=["very_short", "short", "medium", "long", "very_long"],
    )

    df["effective_distance"] = df["distance_km"] * df["traffic_intensity"]
    df["total_workload"] = df["distance_km"] * (df["num_orders"] + 1)
    df["congestion_batch_impact"] = df["traffic_intensity"] * df["num_orders"]
    df["distance_x_condition"] = df["distance_km"] * df["vehicle_condition"]
    df["rush_hour_x_traffic"] = df["is_rush_hour"] * df["traffic_intensity"]
    df["festival_x_traffic"] = df["is_festival_num"] * df["traffic_intensity"]
    df["rider_experience"] = df[f"{NUM_COL}_imputed"] * df["rider_age_imputed"]
    df["weather_severity"] = df[CAT_COL].map(WEATHER_SEVERITY).fillna(2)

    return df


NUMERIC_FEATURES = [
    f"{NUM_COL}_imputed", f"{NUM_COL}_missing",
    "rider_age_imputed", "rider_age_missing",
    "distance_km", "effective_distance", "total_workload",
    "congestion_batch_impact", "distance_x_condition",
    "rush_hour_x_traffic", "festival_x_traffic",
    "rider_experience", "weather_severity",
    "order_hour", "order_min", "is_festival_num",
    "vehicle_condition", "num_orders",
    "day_of_week", "is_weekend",
    "is_rush_hour", "is_late_night", "is_lunch_time", "is_dinner_time",
    "prep_time_min",
]

OHE_COLS = [
    (CAT_COL, CAT_COL),
    ("traffic_density", "traffic"),
    ("vehicle_type", "vehicle"),
    ("city", "city"),
    ("order_type", "order"),
    ("distance_bin", "dist_bin"),
]


def build_feature_matrix(df):
    parts = [df[NUMERIC_FEATURES]]
    for col, prefix in OHE_COLS:
        src = df[col].fillna("meal") if col == "order_type" else df[col]
        parts.append(pd.get_dummies(src, prefix=prefix, dtype=int))
    return pd.concat(parts, axis=1)

## 3. Data Preparation

In [ ]:
train_model = train_df[ALL_FEATURES].copy()

for col in COORDINATE_COLS:
    train_model = train_model[train_model[col] >= 1]

train_model = train_model.dropna(subset=[TARGET_COL]).reset_index(drop=True)

stats = {
    "median_num": train_model[NUM_COL].median(),
    "mode_cat": train_model[CAT_COL].mode()[0],
    "mode_city": train_model["city"].mode()[0],
    "median_rider_age": train_model["rider_age"].median(),
    "time_mode": pd.to_datetime( train_model["order_time"].str.strip(), errors="coerce").mode()[0],
    "date_mode": pd.to_datetime(train_model["order_date"].astype(str).str.strip(), errors="coerce").mode()[0],
}

for k, v in stats.items():
    print(f"{k:20s}: {v}")

In [ ]:
TEST_FEATURES = [f for f in ALL_FEATURES if f != TARGET_COL]
test_model = test_df[TEST_FEATURES].copy()

train_processed = preprocess(train_model, stats, is_train=True)
test_processed = preprocess(test_model,  stats, is_train=False)

X = build_feature_matrix(train_processed)
y = np.log1p(train_processed[TARGET_COL])

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

## 4. Hyperparameter Search

Random search over a grid of XGBoost hyperparameters, evaluated by MAPE on a held-out validation set.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.02, 0.03, 0.05],
    "max_depth": [6, 7, 8],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_weight": [1, 2, 3],
    "gamma": [0, 0.1, 0.2],
    "reg_alpha": [0.0, 0.1],
}

all_combinations = list(product(*param_grid.values()))
random.seed(42)
sampled = random.sample(all_combinations, min(100, len(all_combinations)))
print(f"Evaluating {len(sampled)} randomly sampled hyperparameter combinations...")

In [ ]:
results = []

for idx, values in enumerate(sampled, start=1):
    params = dict(zip(param_grid.keys(), values))

    model = XGBRegressor(
        **params,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
    )
    model.fit(X_train, y_train)

    pred = np.expm1(model.predict(X_valid))
    mape = mean_absolute_percentage_error(np.expm1(y_valid), pred)

    results.append({**params, "valid_mape": mape})

    if idx % 10 == 0 or mape < 0.15:
        print(f"[{idx:03d}/{len(sampled)}] MAPE={mape:.6f} | {params}")

results_df = pd.DataFrame(results).sort_values("valid_mape").reset_index(drop=True)
print("\nTop 5 hyperparameter combinations:")
print(results_df.head())

## 5. Final Model Training

Retrain using the best hyperparameters found above on the full training set.

In [ ]:
best_params = results_df.iloc[0].drop("valid_mape").to_dict()
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_child_weight"] = int(best_params["min_child_weight"])

print("Best parameters:", best_params)
print(f"Best validation MAPE: {results_df.iloc[0]['valid_mape']:.6f}")

In [ ]:
final_model = XGBRegressor(
    **best_params,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
)
final_model.fit(X, y)
print("Final model trained on full dataset.")

## 6. Generate Submission

In [ ]:
X_test = build_feature_matrix(test_processed).reindex(columns=X.columns, fill_value=0)

test_pred = np.expm1(final_model.predict(X_test))
test_pred = np.clip(test_pred, 0, None)

print("Prediction stats:")
print(pd.Series(test_pred).describe())

In [ ]:
try:
    submission = pd.read_csv(SAMPLE_SUB_PATH)
    pred_col = [c for c in submission.columns if c.lower() != "id"][0]
    submission[pred_col] = test_pred
except FileNotFoundError:
    id_col = next(
        (c for c in ["id", "ID", "Id"] if c in test_df.columns), None
    )
    submission = pd.DataFrame(
        {id_col: test_df[id_col], TARGET_COL: test_pred}
        if id_col else {TARGET_COL: test_pred}
    )

submission.to_csv(OUTPUT_SUB_PATH, index=False)
print(f"Saved: {OUTPUT_SUB_PATH}")
submission.head()